In [32]:
here::i_am("atac/archR/dimensionality_reduction/cells/archR_dimensionality_reduction.R")

source(here::here("settings.R"))
source(here::here("utils.R"))

suppressPackageStartupMessages(library(ArchR))

######################
## Define arguments ##
######################

# p <- ArgumentParser(description='')
# p$add_argument('--metadata',        type="character",                               help='Cell metadata file')
# p$add_argument('--matrix',          type="character",  default="PeakMatrix",   help='Matrix to use')
# # p$add_argument('--samples',         type="character",                nargs='+',     help='Samples')
# p$add_argument('--stages',       type="character",  default="all",  nargs='+',  help='Stages to plot')
# p$add_argument('--remove_ExE_cells', action="store_true",                                 help='Remove ExE cells?')
# p$add_argument('--nfeatures',       type="integer",    default=1000,               help='Number of features')
# p$add_argument('--ndims',           type="integer",    default=30,                  help='Number of LSI dimensions')
# p$add_argument('--batch.variable',  type="character",                               help='Metadata column to apply batch correction on')
# p$add_argument('--batch.method',    type="character",  default="MNN",               help='Batch correctin method ("Harmony" or "MNN")')
# p$add_argument('--n_neighbors',     type="integer",    default=30,   nargs='+',     help='(UMAP) Number of neighbours')
# p$add_argument('--min_dist',        type="double",     default=0.3,  nargs='+',     help='(UMAP) Minimum distance')
# p$add_argument('--colour_by',       type="character",  nargs='+',  help='Metadata columns to colour the UMAP by')
# p$add_argument('--seed',            type="integer",    default=42,                  help='Random seed')
# p$add_argument('--outdir',          type="character",                               help='Output directory')

# args <- p$parse_args(commandArgs(TRUE))

## START TEST ##
args <- list()
args$metadata <- "/rds/project/rds-SDzz0CATGms/users/bt392/09_Eomes_invitro_blood/results/atac/archR/qc/sample_metadata_after_qc.txt.gz"
args$nfeatures <- 25000
args$matrix <- "PeakMatrix"
args$vars_to_regress <- c('TSSEnrichment_atac')
# args$outdir <- "/bi/group/reik/ricard/data/gastrulation_multiome_10x/results/atac/archR/dimensionality_reduction"
## END TEST ##



########################
## Load ArchR project ##
########################

source(here::here("atac/archR/load_archR_project.R"))

ERROR: Error: Could not find associated project in working directory or any parent directory.
- Path in project: atac/archR/dimensionality_reduction/cells/archR_dimensionality_reduction.R
- Current working directory: /rds/project/rds-SDzz0CATGms/users/bt392/09_Eomes_invitro_blood/processed/atac/archR
Please open the project associated with this file and try again.


In [71]:
a = Sys.time()
b = Sys.time()

b - a

Time difference of 0.001159906 secs

In [46]:
args$outdir = file.path(io$basedir, 'results/atac/archR/regression/')
dir.create(args$outdir, recursive=TRUE, showWarnings =FALSE)

In [3]:
stats = fread('/rds/project/rds-SDzz0CATGms/users/bt392/09_Eomes_invitro_blood/results/atac/archR/feature_stats/PeakMatrix/PeakMatrix_Clusters_PeakMatrix2_stats.txt.gz')

In [6]:
features = stats[order(-var_pseudobulk)] %>% head(., args$nfeatures) %>% .$feature

In [7]:
head(features)

[1] "chr16:48252720-48253320"  "chr9:60350988-60351588"  
[3] "chr5:75962064-75962664"   "chr7:103909945-103910545"
[5] "chr8:68606563-68607163"   "chr15:12778631-12779231"

In [8]:
PeakMatrix = readRDS('/rds/project/rds-SDzz0CATGms/users/bt392/09_Eomes_invitro_blood/processed/atac/archR/Matrices/PeakMatrix_summarized_experiment.rds')

In [27]:
PeakMatrix

class: RangedSummarizedExperiment 
dim: 238993 35536 
metadata(0):
assays(1): PeakMatrix
rownames(238993): chr1:3003473-3004073 chr1:3035580-3036180 ...
  chrX:169923276-169923876 chrX:169925415-169926015
rowData names(1): idx
colnames(35536): 2_Eo_DEG_G9_day3_5_VC#CTACGAAGTGAGCACT-1
  2_Eo_DEG_G9_day3_5_VC#TGGACAAAGCTCAATA-1 ...
  2_Eo_DEG_G9_day5_VC#CTCCCTGAGGTGAAAT-1
  2_Eo_DEG_G9_day5_VC#CGCTCCATCATCCTGC-1
colData names(38): BlacklistRatio nDiFrags ... ReadsInPeaks FRIP

In [36]:
##########################
## Load sample metadata ##
##########################

sample_metadata <- fread(args$metadata) %>%
  .[,log_nFrags_atac:=log10(nFrags_atac)]

sample_metadata = sample_metadata[match(colnames(PeakMatrix), cell),]

In [38]:
summary(sample_metadata$cell == colnames(PeakMatrix))

   Mode    TRUE 
logical   35536 

In [19]:
peak.mtx = PeakMatrix[features,]@assays@data$PeakMatrix

In [21]:
peak_norm.mtx = tfidf(peak.mtx)

In [42]:
# Regress out variables 
peak_norm_regressed.mtx <- RegressOutMatrix(
    mtx = head(peak_norm.mtx),
    covariates = sample_metadata[,args$vars_to_regress,with=F]
  )

fwrite(peak_norm_regressed.mtx, file.path(args$regression_out,"peak_norm_regressed_mtx.txt.gz"))

In [45]:
head(peak.mtx)

  [[ suppressing 34 column names ‘2_Eo_DEG_G9_day3_5_VC#CTACGAAGTGAGCACT-1’, ‘2_Eo_DEG_G9_day3_5_VC#TGGACAAAGCTCAATA-1’, ‘2_Eo_DEG_G9_day3_5_VC#GTGAGCGAGCTCCTAC-1’ ... ]]



6 x 35536 sparse Matrix of class "dgCMatrix"
                                                                               
[1,] . 2 2 . . . . . . . . . . . . . . . . . . . . . . . 1 . . . . . . . ......
[2,] . . . . . . . . . . . . . . . . . . 1 1 . . . . . . . . . . . . . . ......
[3,] . . 2 . 2 . . . . . . . . . . . . . . . 1 . . . . . . . . . . . . . ......
[4,] . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . ......
[5,] 1 . . . . . . . . . . . . . . . . . . . 3 . . . . . . . . . . . . . ......
[6,] . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . ......

 .....suppressing 35502 columns in show(); maybe adjust 'options(max.print= *, width = *)'
 ..............................

In [43]:
peak_norm_regressed.mtx

2_Eo_DEG_G9_day3_5_VC#CTACGAAGTGAGCACT-1,2_Eo_DEG_G9_day3_5_VC#TGGACAAAGCTCAATA-1,2_Eo_DEG_G9_day3_5_VC#GTGAGCGAGCTCCTAC-1,2_Eo_DEG_G9_day3_5_VC#CATTGTGCATGTTTGG-1,2_Eo_DEG_G9_day3_5_VC#AATCATCCAAGGAATC-1,2_Eo_DEG_G9_day3_5_VC#GGAACTAAGTTCCCGT-1,2_Eo_DEG_G9_day3_5_VC#GTCGGTTCAATTGAAG-1,2_Eo_DEG_G9_day3_5_VC#GGGTCACTCAGGCTAT-1,2_Eo_DEG_G9_day3_5_VC#CTTCGCGTCATGGTGT-1,2_Eo_DEG_G9_day3_5_VC#ACAGCGCTCCGGGACT-1,⋯,2_Eo_DEG_G9_day5_VC#GTTTATCTCATGAGCT-1,2_Eo_DEG_G9_day5_VC#GGTGCTTCAGGACACA-1,2_Eo_DEG_G9_day5_VC#CGTAGCGGTTCCAGGG-1,2_Eo_DEG_G9_day5_VC#TCGTTATTCGCTAGTG-1,2_Eo_DEG_G9_day5_VC#TCGATTAAGCGAAGTA-1,2_Eo_DEG_G9_day5_VC#CGGATAAAGGATTTGC-1,2_Eo_DEG_G9_day5_VC#GACCTTTGTTGCGGAT-1,2_Eo_DEG_G9_day5_VC#TGTAAAGCACATACTG-1,2_Eo_DEG_G9_day5_VC#CTCCCTGAGGTGAAAT-1,2_Eo_DEG_G9_day5_VC#CGCTCCATCATCCTGC-1
-0.2041676,3.1464522,2.4364945,-0.2004072,-0.2034071,-0.2008720,-0.2007030,-0.2059844,-0.2037451,-0.1971539,⋯,-0.1724790,-0.1792815,-0.1721832,-0.1865910,-0.14408591,-0.1640286,-0.1904781,-0.1651694,-0.1807603,5.7628686
-0.2426973,-0.2900880,-0.2467856,-0.2352717,-0.2411955,-0.2361895,-0.2358557,-0.2462850,-0.2418630,-0.2288472,⋯,-0.1801216,-0.1935545,-0.1795376,-0.2079887,-0.12405379,-0.1634348,-0.2156646,-0.1656875,-0.1964747,-0.1682740
-0.2231559,-0.2830752,2.3095762,-0.2137672,2.4553224,-0.2149276,-0.2145056,-0.2276921,-0.2221010,-0.2056443,⋯,-0.1440372,-0.1610214,-0.1432988,-0.1792714,-0.07314687,-0.1229389,5.4682400,6.0377461,-0.1647136,-0.1290574
-0.1877646,-0.2340450,-0.1917571,-0.1805129,-0.1862979,-0.1814091,-0.1810832,-0.1912682,-0.1869498,-0.1742389,⋯,-0.1266548,-0.1397730,-0.1260844,-0.1538690,-0.07190042,-0.1103588,-0.1613651,-0.1125588,-0.1426248,-0.1150847
1.5114402,-0.2619137,-0.2098098,-0.1959556,-0.2030835,-0.1970599,-0.1966584,-0.2092075,-0.2038866,-0.1882254,⋯,-0.1295960,-0.1457593,-0.1288933,-0.1631272,-0.06213210,-0.1095175,-0.1723633,-0.1122281,-0.1492730,-0.1153403
-0.2750568,-0.3458263,-0.2811619,-0.2639679,-0.2728141,-0.2653384,-0.2648400,-0.2804143,-0.2738108,-0.2543741,⋯,-0.1816111,-0.2016708,-0.1807390,-0.2232256,-0.09788381,-0.1566923,-0.2346882,-0.1600563,-0.2060316,-0.1639187


In [9]:
tfidf

function (mtx, method = 1, scale.factor = 10000) 
{
    npeaks <- colSums(mtx)
    if (any(npeaks == 0)) {
        warning("Some cells contain 0 total counts")
    }
    tf <- Matrix::tcrossprod(mtx, y = Matrix::Diagonal(x = 1/npeaks))
    rsums <- rowSums(mtx)
    if (any(rsums == 0)) {
        warning("Some features contain 0 total counts")
    }
    idf <- ncol(mtx)/rsums
    if (method == 2) {
        idf <- log(1 + idf)
    }
    else if (method == 3) {
        tf <- log1p(tf * scale.factor)
        idf <- log(1 + idf)
    }
    mtx.tfidf <- Matrix::Diagonal(n = length(idf), x = idf) %*% 
        tf
    if (method == 1) {
        mtx.tfidf <- log1p(mtx.tfidf * scale.factor)
    }
    colnames(mtx.tfidf) <- colnames(mtx)
    rownames(mtx.tfidf) <- rownames(mtx)
    mtx.tfidf[is.na(mtx.tfidf)] <- 0
    return(mtx.tfidf)
}

In [75]:
args$regression_out = file.path(io$basedir, 'results/atac/archR/regression/')


In [76]:
test = fread(file.path(args$regression_out,"peak_norm_regressed_mtx.txt.gz"))

In [77]:
atac_tfidf.mtx = test
rm(test)

In [80]:
minmax = function(x){
    x = (x-min(x))/(max(x)-min(x))
    return(x)
}

In [88]:
# minmax normalisation by column, maybe should be done by row
atac_tfidf_minmax.mtx = apply(atac_tfidf.mtx, 2, minmax)

In [89]:
svd <- irlba::irlba(atac_tfidf_minmax.mtx, args$ndims, args$ndims)
svdDiag <- matrix(0, nrow=args$ndims, ncol=args$ndims)
diag(svdDiag) <- svd$d
lsi.mtx <- t(svdDiag %*% t(svd$v))
rownames(lsi.mtx) <- colnames(atac_tfidf_minmax.mtx)
colnames(lsi.mtx) <- paste0("LSI",seq_len(ncol(lsi.mtx)))

rm(svd,svdDiag); gc(reset=T)

Warning message in max(nu, nv):
“no non-missing arguments to max; returning -Inf”


ERROR: Error in irlba::irlba(atac_tfidf_minmax.mtx, args$ndims, args$ndims): max(nu, nv) must be positive


In [ ]:
lsi_scaled.mtx = scale(lsi.mtx)